<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
# 5 · The Transformer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/05-transformer-from-scratch.ipynb)

**From Traditional NLP to Modern LLMs** · Day 2 · 120 minutes (55 briefing, 55 lab, 10 debrief)

Self-attention, multi-head attention and positional encodings, assembled into a small GPT.

**By the end of this notebook you can:**

- Implement scaled dot-product attention with a causal mask
- Inspect how heads, blocks and positions assemble into a decoder-only transformer
- Train a small GPT with the provided loop and evaluate it against the LSTM

**Stack:** PyTorch

In [ ]:
#@title Workshop harness: run this cell first { display-mode: "form" }
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Tick WORKED_EXAMPLE to run every reference solution in place of your code. That gives a
# complete worked example; it does not show that you did the exercises.
WORKED_EXAMPLE = False  #@param {type:"boolean"}

import os as _os
import time as _time

from IPython import get_ipython as _get_ipython

_NOTEBOOK = "05-transformer-from-scratch"
_CONTENT_SHA = "3bf87e65364e4c73"
_EXERCISES = {1: ('scaled_dot_product_attention',), 2: ('causal_mask',), 3: ('embed_with_positions',), 5: ('mean_attention_distance',), 6: ('MyMultiHeadAttention', 'MyBlock')}
_HINTS = {1, 2, 5}  # exercises with a folded Hint cell above their TODO cell
_PATHS = {'offline': 'offline stand-ins and test doubles: checks that the code runs, not what a model does', 'open': 'real open models from the Hugging Face Hub, no API keys', 'keyed': 'commercial APIs with keys'}
_FALLBACK_NOTE = None
_RECORDED_SETTINGS = ('NLP_LLMS_OFFLINE_TINY', 'NLP_LLMS_STUB', 'NLP_LLMS_LAB07_OFFLINE', 'NLP_LLMS_LAB09_OFFLINE', 'NLP_LLMS_LAB10_OFFLINE', 'NLP_LLMS_LAB13_OFFLINE', 'NLP_LLMS_LAB14_OFFLINE', 'NLP_LLMS_LAB15_OFFLINE', 'NLP_LLMS_QUICK', 'HF_HUB_OFFLINE')
_OFFLINE_FLAGS = ('NLP_LLMS_OFFLINE_TINY', 'NLP_LLMS_STUB', 'NLP_LLMS_LAB07_OFFLINE', 'NLP_LLMS_LAB09_OFFLINE', 'NLP_LLMS_LAB10_OFFLINE', 'NLP_LLMS_LAB13_OFFLINE', 'NLP_LLMS_LAB14_OFFLINE', 'NLP_LLMS_LAB15_OFFLINE')
_FALLBACK_MARKERS = ('Could not load ', 'did not load', 'USING StubProvider', 'LSA stand-in fitted', 'OFFLINE TEST MODE')
_RECORD_PACKAGES = (
    "torch", "transformers", "numpy", "scikit-learn", "peft", "datasets", "openai",
    "anthropic", "typesafe-sdk", "langgraph", "langchain-core", "llama-index-core",
)


class _Workshop:
    """Keeps the folded solutions from replacing your code, and says whose code each
    checkpoint checked: yours, the reference solution, or code the lab provides."""

    _MISSING = object()

    def __init__(self, old=None):
        # Rerunning this cell (as Run all does) keeps the stored solutions but starts a new
        # run: the checkpoint results, cell times and run clock describe this run only.
        self.ref = getattr(old, "ref", {})
        self.stub = getattr(old, "stub", {})
        self.chosen = getattr(old, "chosen", set())  # exercises switched by use_reference
        self.verified = {}
        self._verifying = None
        self.results = {}
        self.cell_seconds = []
        self.cell_errors = 0
        self.fallbacks = set()
        self.started = _time.time()
        self._current = None
        self._t0 = None

    @property
    def worked(self):
        if _os.environ.get("NLP_LLMS_WORKED") == "1":
            return True
        return bool(_get_ipython().user_ns.get("WORKED_EXAMPLE"))

    def _is_reference(self, n, name, obj):
        return any(obj is r for r in self.ref.get(n, {}).get(name, ()))

    def solution(self, n):
        """Decorator on a reference solution: stores it and, unless WORKED_EXAMPLE is
        ticked, leaves your own definition of the same name in place."""

        def bind(obj, name=None):
            name = name or obj.__name__
            self.ref.setdefault(n, {}).setdefault(name, []).append(obj)
            current = _get_ipython().user_ns.get(name, self._MISSING)
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.stub.setdefault(n, {})[name] = current
            if self.worked:
                return obj
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.chosen.discard(n)  # your TODO cell ran again: back to your code
                return current
            if n in self.chosen:
                return obj  # you chose the reference with use_reference(n)
            # Your TODO cell has not run, or the name still holds a reference from a
            # worked run: go back to your code if it is known, never silently to the reference.
            yours = self.stub.get(n, {}).get(name, self._MISSING)
            if yours is not self._MISSING:
                return yours
            if callable(obj):
                return self._not_run(n, name)
            print(f"[workshop] Run your # TODO {n} cell: {name} is the reference until you do.")
            return obj

        return bind

    def _not_run(self, n, name):
        """Stands in for a function or class whose TODO cell has not run yet."""

        def missing(*args, **kwargs):
            raise NotImplementedError(
                f"{name}: run your # TODO {n} cell first, or go on with"
                f" workshop.use_reference({n!r})"
            )

        missing.__name__ = name
        return missing

    def solution_value(self, n, name, value):
        """The same as solution(n), for a value rather than a function or class."""
        if value is None or isinstance(value, (bool, int, float, str, bytes)):
            # Python shares equal small values, so yours and the reference could not be told
            # apart: give such a value in the stub instead of making it part of the exercise.
            kind = type(value).__name__
            raise TypeError(f"solution_value({n!r}, {name!r}): use an object, not a plain {kind}")
        return self.solution(n)(value, name)

    def use_reference(self, n):
        """Go on with the reference solution of exercise n in place of yours."""
        if n not in self.ref:
            raise KeyError(f"Exercise {n} has no reference solution yet: run its Solution cell")
        ns = _get_ipython().user_ns
        for name, objs in self.ref[n].items():
            ns[name] = objs[-1]
        self.chosen.add(n)
        print(
            f"Exercise {n}: now using the REFERENCE solution. Rerun the cells below it."
            f" To switch back to your code, rerun your # TODO {n} cell."
        )

    def checkpoint(self, n=None, label=None):
        """First line of a checkpoint cell: notes whose code the cell is about to check."""
        names = _EXERCISES.get(n, ())
        ns = _get_ipython().user_ns
        k = sum(self._is_reference(n, x, ns.get(x, self._MISSING)) for x in names)
        if not names:
            whose = None  # the results of the cells above: provided code, maybe your earlier work
        elif k == len(names):
            whose = "the REFERENCE solution"
        elif k == 0:
            whose = "your code"
        else:
            whose = "partly the REFERENCE solution"
        self._current = (n, str(label if label is not None else n), whose)

    def _pre(self, info):
        self._t0 = _time.time()

    def _post(self, result):
        if self._t0 is None:  # this harness cell itself: its hooks were registered mid-cell
            return
        self.cell_seconds.append(round(_time.time() - self._t0, 2))
        if not result.success and self._verifying is None:
            self.cell_errors += 1  # a verification copy fails on purpose; it is not an error
        if self._current is None:
            return
        n, label, whose = self._current
        self._current = None
        if self._verifying is not None:
            self.verified[self._verifying] = bool(result.success)
            return
        self.results[label] = (bool(result.success), whose)
        if whose is None:
            if result.success:
                print(f"[workshop] Checkpoint {label} passed.")
            elif isinstance(result.error_in_exec, NotImplementedError):
                print(
                    f"[workshop] Checkpoint {label} uses an exercise you have not written yet."
                    " Finish it, or go on with workshop.use_reference(N) for that exercise."
                )
            else:
                print(f"[workshop] Checkpoint {label} failed. Read the message above.")
        elif result.success:
            print(f"[workshop] Checkpoint {label} passed on {whose}.")
        elif isinstance(result.error_in_exec, NotImplementedError):
            print(
                f"[workshop] Checkpoint {label}: TODO {n} is not written yet. Write it, run"
                " its cell, then rerun this one. To go on without it:"
                f" workshop.use_reference({n!r})"
            )
            if n in _HINTS:
                print(f"[workshop] Stuck? Open the folded Hint above TODO {n} first.")
        elif whose == "the REFERENCE solution":
            print(
                f"[workshop] Checkpoint {label} failed on the reference solution: a problem"
                " with the lab or the runtime, not with your code. Tell the instructor."
            )
        else:
            print(
                f"[workshop] Checkpoint {label} failed on {whose}. Read the assertion above,"
                f" fix TODO {n}, or go on with workshop.use_reference({n!r})."
            )
            if n in _HINTS:
                print(f"[workshop] The folded Hint above TODO {n} says what to check.")

    def _verify_begin(self, n):
        """Test-only (scripts/test_notebooks.py --verify-checkpoints): bind exercise n's
        stubs, so that a copy of its checkpoint runs on unfinished code."""
        ns = _get_ipython().user_ns
        self._saved = {name: ns.get(name, self._MISSING) for name in _EXERCISES.get(n, ())}
        ns.update(self.stub.get(n, {}))
        self._verifying = n

    def _verify_end(self, n):
        """Test-only: restore the reference and require that the stub copy failed."""
        ns = _get_ipython().user_ns
        for name, value in self._saved.items():
            if value is self._MISSING:
                ns.pop(name, None)
            else:
                ns[name] = value
        self._verifying = None
        if self.verified.get(n) is not False:
            raise AssertionError(
                f"The checkpoint of exercise {n} passed on the unfinished stub: it cannot tell"
                " a finished exercise from an unfinished one"
            )

    def summary(self):
        """Which checkpoints passed, and on whose code."""
        groups = {"your code": [], "the reference": [], "other checks": [], "failed": []}
        for label, (ok, whose) in self.results.items():
            if not ok:
                groups["failed"].append(label)
            elif whose is None:
                groups["other checks"].append(label)
            elif whose == "your code":
                groups["your code"].append(label)
            else:
                groups["the reference"].append(label)
        mode = "WORKED EXAMPLE: reference solutions" if self.worked else "your code"
        print(f"{_NOTEBOOK}, run with {mode}.")
        path = self.path()
        # _PATHS comes from readiness.paths in _variables.yml; the offline entry says that
        # such numbers check the code, not a model.
        print(f"  What ran: {_PATHS[path]}.")
        if path == "open" and _FALLBACK_NOTE:
            print(f"  Read the numbers with this in mind: {_FALLBACK_NOTE}.")
        lines = {
            "your code": "Checkpoints passed on your code",
            "the reference": "Checkpoints passed on a reference solution",
            "other checks": "Other checks passed",
            "failed": "Checkpoints failed",
        }
        for group, labels in groups.items():
            print(f"  {lines[group]}: {', '.join(labels) or 'none'}")
        if self.worked:
            print("  A worked-example run shows how the lab goes, not that you did it.")

    def path(self):
        """offline, keyed or open: the same rule as scripts/add_run_record.py's path_of."""
        ns = _get_ipython().user_ns
        if self.fallbacks or any(_os.environ.get(flag) for flag in _OFFLINE_FLAGS):
            return "offline"
        if ns.get("PROVIDER") in ("openai", "anthropic") or ns.get("JEV_PATH") == "keyed":
            return "keyed"
        return "open"

    def run_record(self):
        """A run record for runs/ (see runs/README.md): paste it into
        scripts/add_run_record.py. It holds no code, outputs, keys or names."""
        import importlib.metadata as _md
        import json
        import platform
        import sys

        gpu = None
        try:
            import torch

            if torch.cuda.is_available():
                gpu = torch.cuda.get_device_name(0)
        except Exception:
            pass
        packages = {}
        for name in _RECORD_PACKAGES:
            try:
                packages[name] = _md.version(name)
            except Exception:
                pass
        import datetime as _dt

        record = {
            "date": _dt.datetime.now(_dt.timezone.utc).date().isoformat(),
            "notebook": _NOTEBOOK,
            "content_sha": _CONTENT_SHA,
            "mode": "worked" if self.worked else "learner",
            # A run passes when it reached this last cell with no error on the way and every
            # checkpoint passed. Run all reruns the harness first, so it starts clean.
            "status": "pass"
            if self.results and self.cell_errors == 0 and all(ok for ok, _ in self.results.values())
            else "fail",
            "cell_errors": self.cell_errors,
            "fallbacks": sorted(self.fallbacks),
            "seconds": round(_time.time() - self.started, 1),
            "cell_seconds": self.cell_seconds,
            "python": platform.python_version(),
            "platform": sys.platform,
            "gpu": gpu,
            "colab_release": _os.environ.get("COLAB_RELEASE_TAG"),
            "provider": _get_ipython().user_ns.get("PROVIDER"),
            "jev_path": _get_ipython().user_ns.get("JEV_PATH"),
            "settings": {k: _os.environ[k] for k in _RECORDED_SETTINGS if _os.environ.get(k)},
            "packages": packages,
        }
        print("----- run record (paste into scripts/add_run_record.py) -----")
        print(json.dumps(record))
        print("----- end of run record -----")


_ip = _get_ipython()
_old = _ip.user_ns.get("workshop")
# Remove only an earlier harness's own hooks; every other hook (Colab's, an extension's)
# stays registered.
for _event in ("pre_run_cell", "post_run_cell"):
    for _callback in list(_ip.events.callbacks[_event]):
        if type(getattr(_callback, "__self__", None)).__name__ == "_Workshop":
            _ip.events.unregister(_event, _callback)
workshop = _Workshop(_old if hasattr(_old, "ref") else None)

# Notice when a lab falls back to a stand-in, for the run record. print is wrapped once.
import builtins as _builtins

if not getattr(_builtins.print, "_workshop", False):
    _print = _builtins.print

    def _watching_print(*args, **kwargs):
        try:
            ip = _get_ipython()
            ws = ip.user_ns.get("workshop") if ip is not None else None
            if ws is not None and hasattr(ws, "fallbacks"):
                # Labs print their fallback notices as strings; never stringify other objects.
                text = " ".join(a for a in args if isinstance(a, str))
                ws.fallbacks.update(m.strip() for m in _FALLBACK_MARKERS if m in text)
        except Exception:
            pass  # watching must never break printing
        return _print(*args, **kwargs)

    _watching_print._workshop = True
    _builtins.print = _watching_print
_ip.events.register("pre_run_cell", workshop._pre)
_ip.events.register("post_run_cell", workshop._post)
print("Workshop harness ready:", "WORKED EXAMPLE" if workshop.worked else "checking your code")


## Setup

This lab is designed for a **T4 GPU runtime** (Runtime > Change runtime type > T4 GPU). It needs no API key and no installs: PyTorch, NumPy and matplotlib are preinstalled on Colab.

- **On a GPU** the models are trained with the settings the lab was designed around (batch 64, 3,000 steps each), intended to fit the 10-minute compute target of a free T4. That has not yet been measured on a T4: the reference run of these settings was on a CPU.
- **Without a GPU** the setup cell sets `QUICK = True`: the same models, trained on batches of 16 for 1,500 steps, so that Run all finishes on a CPU. The checkpoints still pass, but the results table will not match the GPU numbers. As in Lab 3, the environment variable `NLP_LLMS_QUICK` (`1` or `0`) forces the choice.

**How the lab works.** There are five exercises and one optional stretch section. Each exercise has:

1. a **Predict** question to answer before you run anything;
2. a `# TODO` cell where you write one short function;
3. a folded **Solution** cell (open it only after trying);
4. a **Checkpoint** cell that tells you whether your function is right.

The folded solution cells never replace your code: each checkpoint checks your own functions and says so. If you are stuck, `workshop.use_reference(N)` lets you go on with exercise N's reference solution, and the checkpoint then says it checked the reference. To run the whole lab as a worked example, tick `WORKED_EXAMPLE` in the harness cell at the top; a worked run shows how the lab goes, not that you did it.

**What you write and what is provided.** You write the attention step, the causal mask, the input step of the model and one attention statistic. The multi-head layer, the transformer block, the model class, the training loop, the evaluation and the plots are provided. Writing the multi-head split and the block yourself is the stretch section.

In [ ]:
# Setup: imports, seeds, device and mode. Everything here is preinstalled on Colab.
import json
import math
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# QUICK = True trains the same models on smaller batches for fewer steps: for a CPU runtime or a
# dry run. Its numbers are worse than the lab's. It is on by default when there is no GPU;
# NLP_LLMS_QUICK=1 or NLP_LLMS_QUICK=0 (the flag Lab 3 uses) forces it on or off.
_quick_flag = os.environ.get("NLP_LLMS_QUICK")
QUICK = _quick_flag == "1" if _quick_flag in ("0", "1") else DEVICE == "cpu"

NAVY, ACCENT = "#16324f", "#b3541e"  # plot colors
LAB_START = time.time()
TIMES = {}  # section -> seconds, printed at the end of the core path

print(f"PyTorch {torch.__version__} · device {DEVICE} · QUICK = {QUICK} · {torch.get_num_threads()} CPU threads")
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
elif QUICK:
    print("No GPU found, so training is shortened (QUICK = True). For the full lab choose\n"
          "Runtime > Change runtime type > T4 GPU, then Run all.")

## Data

The language-model corpus of Labs 1 and 3: **Tiny Shakespeare**, split by position into train, validation and test (1,000,000 / 55,000 / 60,394 characters). The tokens are characters and the vocabulary is the 65 characters of the training split, exactly as in Lab 3, so the numbers in this lab can be put beside Lab 1's and Lab 3's.

The next cell is the workshop's standard loading cell, the same in every lab (see `data/README.md` in the repository). It checks the file against a hash, so everyone gets the same bytes and the same splits.

The rule for the whole workshop: fit on **train**, choose settings on **val**, report on **test** once per model.

In [ ]:
import csv, gzip, hashlib, io, os, urllib.request
from pathlib import Path


def fetch(name, urls, sha256):
    """Return the bytes of a workshop data file, checked against its SHA-256."""
    cache = Path(os.environ.get("NLP_LLMS_DATA", "data"))
    for path in (cache / name, Path("../data") / name):
        if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == sha256:
            return path.read_bytes()
    for url in urls:  # canonical URL first, fallback second
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                blob = response.read()
        except OSError as error:
            print(f"Could not fetch {url}: {error}")
            continue
        if hashlib.sha256(blob).hexdigest() != sha256:
            print(f"Ignoring {url}: the file does not match the expected hash")
            continue
        cache.mkdir(parents=True, exist_ok=True)
        (cache / name).write_bytes(blob)
        return blob
    raise RuntimeError(f"Could not load {name} from any source")


TOPIC_CLASSES = ["cs.CL", "cs.CV", "cs.LG", "cs.RO"]  # label 0, 1, 2, 3
TOPIC_NAMES = ["Language", "Vision", "Machine learning", "Robotics"]


def load_topics():
    """arXiv Topics v1. Returns {"train" | "val" | "test": (texts, labels)}."""
    blob = fetch(
        "arxiv_topics_v1.csv.gz",
        [
            "https://raw.githubusercontent.com/project-delphi/nlp-llms/main/data/arxiv_topics_v1.csv.gz",
            "https://cdn.jsdelivr.net/gh/project-delphi/nlp-llms@main/data/arxiv_topics_v1.csv.gz",
        ],
        "49b51502fc6cf6ecff23fa266c5372594a88414dd96513579237792710246dc5",
    )
    splits = {"train": ([], []), "val": ([], []), "test": ([], [])}
    for row in csv.DictReader(io.StringIO(gzip.decompress(blob).decode("utf-8"))):
        texts, labels = splits[row["split"]]
        texts.append(row["title"] + "\n" + row["abstract"])
        labels.append(int(row["label"]))
    return splits


def load_lm_corpus():
    """Tiny Shakespeare. Returns {"train" | "val" | "test": str}, split by character offset."""
    blob = fetch(
        "tinyshakespeare.txt",
        [
            "https://raw.githubusercontent.com/karpathy/char-rnn/6f9487a6fe5b420b7ca9afb0d7c078e37c1d1b4e/data/tinyshakespeare/input.txt",
            "https://raw.githubusercontent.com/project-delphi/nlp-llms/main/data/tinyshakespeare.txt",
        ],
        "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed",
    )
    text = blob.decode("utf-8")
    return {"train": text[:1_000_000], "val": text[1_000_000:1_055_000], "test": text[1_055_000:]}

In [ ]:
corpus = load_lm_corpus()  # {"train" | "val" | "test": str}
chars = sorted(set(corpus["train"]))
char_to_id = {ch: i for i, ch in enumerate(chars)}
VOCAB_SIZE = len(chars)  # |V| = 65
assert VOCAB_SIZE == 65
assert set(corpus["val"]) <= set(chars) and set(corpus["test"]) <= set(chars), "unseen character"

ids = {split: torch.tensor([char_to_id[ch] for ch in text]) for split, text in corpus.items()}
all_ids = torch.cat([ids["train"], ids["val"], ids["test"]])  # the splits are contiguous
SPLIT_START = {"train": 0, "val": 1_000_000, "test": 1_055_000}  # offset of each split in all_ids

for split, t in ids.items():
    print(f"{split:<5} {len(t):>9,} characters")
print(f"|V| = {VOCAB_SIZE}")
print("\nStart of the test split:\n" + corpus["test"][:160])

### Tools Restated From Lab 3

Notebooks cannot import from each other on Colab, so the two functions of Lab 3 that the comparison depends on are restated here, unchanged (they are the solutions of Lab 3's Exercises 2 and 5):

- `lm_loss_and_ppl(logits, targets)`: the mean negative log-likelihood in **nats per character** (the cross-entropy, briefing eq. `loss`), the perplexity $\mathrm{PPL} = \exp(\text{mean NLL})$, and bits per character $\mathrm{BPC} = \text{mean NLL} / \ln 2$.
- `sample_with_temperature(logits, tau, generator)`: draw the next character from $\mathrm{softmax}(z/\tau)$.

`get_batch` cuts random training windows of `T` characters; the target of each position is the next character (teacher forcing, Module 3).

In [ ]:
def lm_loss_and_ppl(logits, targets):
    """Mean NLL in nats, perplexity and bits per character. logits (..., |V|), targets (...)."""
    logits = logits.reshape(-1, logits.shape[-1])  # (N, |V|): one row per predicted position
    targets = targets.reshape(-1)  # (N,)
    log_probs = F.log_softmax(logits, dim=-1)  # log ŷ_t, (N, |V|)
    loss = -log_probs.gather(1, targets[:, None]).mean()  # mean of -log ŷ_t[w_{t+1}], nats
    return loss, loss.exp(), loss / math.log(2)


def sample_with_temperature(logits, tau, generator=None):
    """One draw per row of `logits` (..., |V|) from softmax(logits / tau). Returns IDs, shape (...)."""
    probs = torch.softmax(logits / tau, dim=-1)  # ŷ^(τ), (..., |V|)
    flat = probs.reshape(-1, probs.shape[-1])  # (N, |V|): multinomial takes 1-D or 2-D input
    draws = torch.multinomial(flat, num_samples=1, generator=generator)  # (N, 1)
    return draws.reshape(probs.shape[:-1])  # (...)


def get_batch(split_ids, batch_size, T, generator):
    """Random windows of a split. x: (B, T) inputs; y: (B, T), the next character of each input."""
    starts = torch.randint(0, len(split_ids) - T - 1, (batch_size,), generator=generator)
    x = torch.stack([split_ids[s : s + T] for s in starts.tolist()])
    y = torch.stack([split_ids[s + 1 : s + T + 1] for s in starts.tolist()])
    return x.to(DEVICE), y.to(DEVICE)


_nll, _ppl, _bpc = lm_loss_and_ppl(torch.zeros(10, VOCAB_SIZE), torch.zeros(10, dtype=torch.long))
assert abs(_ppl.item() - VOCAB_SIZE) < 1e-3, "uniform logits must give PPL = |V|"
print(f"Uniform model: {_nll.item():.4f} nats = ln 65, PPL {_ppl.item():.1f}, {_bpc.item():.4f} bits per character")

# Part A · Building the Model

## Exercise 0 · Read the Model (3 minutes)

The next cell is the whole model, provided. It is the decoder-only transformer of the briefing (eq. `gpt`), pre-norm (eq. `block`), with the briefing's names:

| Briefing | Code |
|---|---|
| $W_Q, W_K, W_V$ (all heads side by side), $W_{\mathrm{proj}}, b_{\mathrm{proj}}$ | `W_Q`, `W_K`, `W_V`, `proj` in `MultiHeadAttention` |
| $n_h$ heads of width $d_k = d / n_h$ | `n_h`, `d_k` |
| $\mathrm{LN}$, $\mathrm{FFN}$ with $d_{\mathrm{ff}} = 4d$ and GELU | `ln1`, `ln2`, `ffn` in `Block` |
| $e_w$, learned $p_t$, $T_{\max}$ | `tok_emb`, `pos_emb`, `T_max` in `GPT` |
| final $\mathrm{LN}$, then $W_o, b_o$ | `ln_f`, `W_o` |

PyTorch's `nn.Linear(d, d)` computes $H W^\top + b$ on the rows of $H$, so `self.W_Q(H)` is $H W_Q$ for all heads at once.

**Task.** Read the cell and find the three places marked `<- Exercise`. They call functions that do not exist yet: `scaled_dot_product_attention` (Exercise 1), `causal_mask` (Exercise 2) and `embed_with_positions` (Exercise 3). Python looks these names up when the model runs, not when the class is defined, so once you have written them the model uses your versions.

**Predict.** `MultiHeadAttention` calls your attention function with `Q` of shape `(B, n_h, T, d_k)`. Which dimensions of `Q` index *positions*, and which only index independent copies of the computation?

In [ ]:
class MultiHeadAttention(nn.Module):
    """Masked multi-head self-attention, briefing eq. mha. Calls YOUR attention function."""

    def __init__(self, d, n_h, dropout=0.0):
        super().__init__()
        assert d % n_h == 0, "d must be divisible by n_h"
        self.n_h, self.d_k = n_h, d // n_h
        self.W_Q = nn.Linear(d, d, bias=False)  # the n_h matrices W_Q^(r), side by side
        self.W_K = nn.Linear(d, d, bias=False)
        self.W_V = nn.Linear(d, d, bias=False)
        self.proj = nn.Linear(d, d)  # W_proj and b_proj
        self.dropout = nn.Dropout(dropout)

    def forward(self, H, mask):
        B, T, d = H.shape  # H: (B, T, d), one row per position
        Q = self.W_Q(H).view(B, T, self.n_h, self.d_k).transpose(1, 2)  # (B, n_h, T, d_k)
        K = self.W_K(H).view(B, T, self.n_h, self.d_k).transpose(1, 2)  # (B, n_h, T, d_k)
        V = self.W_V(H).view(B, T, self.n_h, self.d_k).transpose(1, 2)  # (B, n_h, T, d_k)
        V_bar, A = scaled_dot_product_attention(Q, K, V, mask)  # <- Exercise 1 (and 2: mask)
        V_bar = V_bar.transpose(1, 2).reshape(B, T, d)  # concatenate the heads: (B, T, d)
        return self.dropout(self.proj(V_bar)), A  # A: (B, n_h, T, T), kept for plotting


class Block(nn.Module):
    """One pre-norm transformer block, briefing eq. block."""

    def __init__(self, d, n_h, d_ff, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = MultiHeadAttention(d, n_h, dropout)
        self.ln2 = nn.LayerNorm(d)
        self.ffn = nn.Sequential(  # eq. ffn, applied to each position separately
            nn.Linear(d, d_ff), nn.GELU(), nn.Linear(d_ff, d), nn.Dropout(dropout)
        )

    def forward(self, H, mask):
        attn_out, A = self.attn(self.ln1(H), mask)
        H = H + attn_out  # H' = H + MHA(LN(H))
        H = H + self.ffn(self.ln2(H))  # H_out = H' + FFN(LN(H'))
        return H, A


class GPT(nn.Module):
    """Decoder-only transformer language model, briefing eq. gpt."""

    def __init__(self, vocab_size, d=128, n_h=4, n_l=4, T_max=128, d_ff=512, dropout=0.1):
        super().__init__()
        self.T_max = T_max
        self.tok_emb = nn.Embedding(vocab_size, d)  # e_w, one row per character
        self.pos_emb = nn.Embedding(T_max, d)  # p_t, learned, one row per position
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d, n_h, d_ff, dropout) for _ in range(n_l)])
        self.ln_f = nn.LayerNorm(d)  # final LN (pre-norm leaves the last block unnormalized)
        self.W_o = nn.Linear(d, vocab_size)  # output layer W_o, b_o
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):  # GPT-2 style: small normal weights, zero biases
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, idx, return_attention=False):
        B, T = idx.shape  # idx: (B, T) character IDs
        assert T <= self.T_max, f"the context length is {self.T_max}"
        H = embed_with_positions(idx, self.tok_emb, self.pos_emb)  # <- Exercise 3: (B, T, d)
        H = self.drop(H)
        mask = causal_mask(T).to(idx.device)  # <- Exercise 2: (T, T) bool
        attention = []  # one (B, n_h, T, T) weight tensor per layer
        for block in self.blocks:
            H, A = block(H, mask)
            attention.append(A)
        logits = self.W_o(self.ln_f(H))  # z_t for every position: (B, T, |V|)
        return (logits, attention) if return_attention else logits


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

**Explain.** In `(B, n_h, T, d_k)` only the third dimension indexes positions; `d_k` indexes coordinates of one query. The batch `B` and the head `n_h` are independent copies of the same computation, so your attention function must treat every dimension before the last two as a batch dimension. That is why Exercise 1 tests 3-D and 4-D inputs.

## Exercise 1 · Scaled Dot-Product Attention (12 minutes)

Stack the queries, keys and values of one sequence as the rows of $Q, K, V \in \mathbb{R}^{T \times d_k}$. Then (briefing eqs. `score` and `sdpa`)

$$
A = \mathrm{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right), \qquad \bar{V} = A\, V ,
$$

with the softmax applied to each row. Entry $(t, i)$ of $A$ is $\alpha_{t,i}$, the weight with which position $t$ reads position $i$; each row sums to 1.

An optional boolean `mask` of shape `(T, T)` says which entries are allowed: `True` where position $t$ may read position $i$. Where it is `False`, set the score to $-\infty$ before the softmax, so that the weight is exactly 0 (Exercise 2 builds the mask).

**Predict.** For one sequence with $T = 5$ and $d_k = 8$, what are the shapes of $Q K^\top$, $A$ and $\bar{V}$? What changes if there is a leading batch dimension and a head dimension?

**Task.** Complete `scaled_dot_product_attention(Q, K, V, mask=None)`. It must work for any number of leading dimensions, so transpose only the last two dimensions of `K` (`K.transpose(-2, -1)`). Use `masked_fill` with `float("-inf")`, not a large negative number.

<details>
<summary>Hint for TODO 1</summary>

The softmax runs over the last axis, the keys being read (briefing eq. `sdpa`): if each row sums to 1 over the queries instead, `A.sum(-1)` will not be all ones. Divide the scores by $\sqrt{d_k}$, not by $d_k$ (briefing eq. `variance`). For 3-D and 4-D inputs transpose only the last two dimensions of `K`; `K.T` reverses all of them.

</details>

In [ ]:
# TODO 1: complete this function.
def scaled_dot_product_attention(Q, K, V, mask=None):
    """softmax(Q K^T / sqrt(d_k)) V on the last two dimensions; any leading dimensions.

    Q, K, V: (..., T, d_k). mask: optional (T, T) bool, True where position t may read position i.
    Returns (V_bar, A) with shapes (..., T, d_k) and (..., T, T).
    """
    d_k = Q.size(-1)
    scores = ...  # (..., T, T)
    if mask is not None:
        scores = ...  # -inf wherever mask is False
    A = ...  # softmax over the last dimension
    V_bar = ...  # (..., T, d_k)
    return V_bar, A

In [ ]:
#@title Solution 1 — try it yourself first { display-mode: "form" }
@workshop.solution(1)
def scaled_dot_product_attention(Q, K, V, mask=None):
    """softmax(Q K^T / sqrt(d_k)) V on the last two dimensions; any leading dimensions.

    Q, K, V: (..., T, d_k). mask: optional (T, T) bool, True where position t may read position i.
    Returns (V_bar, A) with shapes (..., T, d_k) and (..., T, T).
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # (..., T, T), entry (t, i) is q_t . k_i / sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))  # exp(-inf) = 0 after the softmax
    A = torch.softmax(scores, dim=-1)  # each row sums to 1
    V_bar = A @ V  # (..., T, d_k), row t is sum_i alpha_{t,i} v_i
    return V_bar, A

<details>
<summary>Why this solution works</summary>

`Q @ K.transpose(-2, -1)` is the matrix of all dot products $q_t^\top k_i$ at once (briefing eq. `sdpa`). The `@` operator treats every dimension before the last two as a batch dimension, so the same line works for `(T, d_k)`, `(B, T, d_k)` and `(B, n_h, T, d_k)`. That is how the multi-head layer runs all heads in one call.

The softmax is over the last dimension, the positions $i$ being read, so each row $t$ is a distribution. `A @ V` then forms each output row as the weighted average $\sum_i \alpha_{t,i} v_i$ (briefing eq. `self-attn`).

`masked_fill(~mask, -inf)` puts $-\infty$ where reading is not allowed. Since $e^{-\infty} = 0$ exactly, those weights are exactly zero and the remaining weights in the row still sum to 1. A large negative number such as $-10^9$ would leave a tiny nonzero weight; `-inf` leaves none.

</details>

In [ ]:
# Checkpoint 1
workshop.checkpoint(1)
def _reference_attention(Q, K, V, mask=None):
    """Explicit loops for one sequence: Q, K, V of shape (T, d_k)."""
    T, d_k = Q.shape
    V_bar, A = torch.zeros_like(V), torch.zeros(T, T)
    for t in range(T):
        scores = torch.stack([Q[t] @ K[i] / math.sqrt(d_k) for i in range(T)])
        if mask is not None:
            scores = torch.where(mask[t], scores, torch.tensor(float("-inf")))
        weights = torch.exp(scores - scores.max())
        A[t] = weights / weights.sum()
        V_bar[t] = (A[t][:, None] * V).sum(dim=0)
    return V_bar, A


_g = torch.Generator().manual_seed(0)
_tril = torch.tril(torch.ones(5, 5, dtype=torch.bool))  # a causal mask, for this check only
for _shape in [(2, 5, 8), (2, 3, 5, 8)]:  # (B, T, d_k) and (B, n_h, T, d_k)
    _Q, _K, _V = (torch.randn(*_shape, generator=_g) for _ in range(3))
    for _mask in (None, _tril):
        _out, _A = scaled_dot_product_attention(_Q, _K, _V, _mask)
        assert _out.shape == _shape, f"V_bar has shape {tuple(_out.shape)}, expected {_shape}"
        assert _A.shape == _shape[:-1] + (5,), f"A has shape {tuple(_A.shape)}"
        assert torch.allclose(_A.sum(-1), torch.ones(_shape[:-1]), atol=1e-6), (
            f"rows of A must sum to 1 over the keys (the last axis); got row sums "
            f"{[round(v, 4) for v in _A.sum(-1).flatten()[:4].tolist()]}: did you softmax over the wrong axis?")
        _flat = [t.reshape(-1, 5, 8) for t in (_Q, _K, _V, _out)]
        for _j in range(_flat[0].shape[0]):  # every (batch, head) slice against the loops
            _ref_out, _ref_A = _reference_attention(_flat[0][_j], _flat[1][_j], _flat[2][_j], _mask)
            assert torch.allclose(_flat[3][_j], _ref_out, atol=1e-5), (
                f"V_bar differs from the reference by up to {(_flat[3][_j] - _ref_out).abs().max().item():.2e}: "
                "divide the scores by sqrt(d_k), not d_k, and return A @ V")
            assert torch.allclose(_A.reshape(-1, 5, 5)[_j], _ref_A, atol=1e-5), "A differs from the reference"
        if _mask is not None:
            assert (_A.triu(1) == 0).all(), "masked weights must be exactly 0: use float('-inf')"
            # PyTorch's fused version; for a boolean attn_mask, True means "may attend".
            _torch_out = F.scaled_dot_product_attention(_Q, _K, _V, attn_mask=_mask)
            assert torch.allclose(_out, _torch_out, atol=1e-5), "differs from F.scaled_dot_product_attention"
print("Checkpoint 1 passed: shapes, row sums and values match the reference for 3-D and 4-D inputs")

**Run: why divide by $\sqrt{d_k}$.** The briefing argued (eq. `variance`) that if the coordinates of $q$ and $k$ are independent with mean 0 and variance 1, then $q^\top k$ has variance $d_k$, and $q^\top k / \sqrt{d_k}$ has variance 1 whatever $d_k$ is.

**Predict.** For $d_k = 16$, $64$ and $256$, write down the standard deviation you expect of the unscaled scores and of the scaled scores. Then: for a row of 128 unscaled scores at $d_k = 256$, what will the largest softmax weight in the row be, roughly?

The cell measures both on random unit-variance vectors, which is the situation at initialization.

In [ ]:
_g = torch.Generator().manual_seed(SEED)
score_std = {}  # d_k -> (std unscaled, std scaled), measured
print(" d_k  sqrt(d_k)  std of q.k  std of q.k/sqrt(d_k)  largest weight in a row of 128: unscaled  scaled")
for d_k in (16, 64, 256):
    q = torch.randn(20_000, d_k, generator=_g)
    k = torch.randn(20_000, d_k, generator=_g)
    raw = (q * k).sum(dim=-1)  # 20,000 dot products q^T k
    score_std[d_k] = (raw.std().item(), (raw / math.sqrt(d_k)).std().item())
    # 256 rows of 128 scores each, as in one attention layer with T = 128.
    Qr, Kr = torch.randn(256, d_k, generator=_g), torch.randn(128, d_k, generator=_g)
    max_raw = torch.softmax(Qr @ Kr.T, dim=-1).max(dim=-1).values.mean().item()
    max_scaled = torch.softmax(Qr @ Kr.T / math.sqrt(d_k), dim=-1).max(dim=-1).values.mean().item()
    print(f"{d_k:>4}  {math.sqrt(d_k):>9.1f}  {score_std[d_k][0]:>10.2f}  {score_std[d_k][1]:>20.3f}"
          f"  {max_raw:>40.3f}  {max_scaled:>6.3f}")
print("A uniform row of 128 weights would have largest weight 1/128 =", round(1 / 128, 4))

for d_k, (raw_std, scaled_std) in score_std.items():
    assert abs(raw_std / math.sqrt(d_k) - 1) < 0.05 and abs(scaled_std - 1) < 0.05

**Explain.** The unscaled standard deviation tracks $\sqrt{d_k}$ (4, 8, 16), and after scaling it is 1 for every $d_k$: eq. `variance`, measured. The last two columns show why it matters. Unscaled, a row of 128 scores at $d_k = 256$ puts most of its weight on one position: the softmax is close to one-hot before any training, and a one-hot softmax passes almost no gradient. Scaled, the largest weight stays small at every $d_k$ and the model starts with soft attention it can learn to sharpen. This is a statement about initialization only; trained models learn peaked attention where it helps.

## Exercise 2 · The Causal Mask (8 minutes)

The model is trained with teacher forcing: one forward pass over $w_1, \dots, w_T$ makes all $T$ predictions, and the prediction at position $t$ is scored against $w_{t+1}$. If position $t$ could read position $t + 1$, it could copy the answer. The causal mask (briefing eq. `mask`) forbids reading ahead:

$$
M_{t,i} = \begin{cases} 0 & i \le t \\ -\infty & i > t \end{cases}, \qquad A = \mathrm{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}} + M\right).
$$

In code the mask is boolean, `True` where reading is allowed, and your Exercise 1 function turns `False` into $-\infty$.

**Predict.** Change the character at position $t_0 = 6$ of a sequence of length 10. Which outputs of a causally masked attention layer can change: positions 0 to 5, positions 6 to 9, or all of them? By how much should the others change?

**Task.** Complete `causal_mask(T)`: a `(T, T)` boolean tensor that is `True` on and below the diagonal. One call to `torch.tril` is enough.

<details>
<summary>Hint for TODO 2</summary>

Row $t$ may read columns $0, \dots, t$, the diagonal included (briefing eq. `mask`): that is the lower triangle. With the upper triangle the layer reads ahead, so changing position 6 changes the outputs at positions 0 to 5; without the diagonal, row 0 has no allowed entry and its softmax is NaN.

</details>

In [ ]:
# TODO 2: complete this function.
def causal_mask(T):
    """(T, T) bool tensor, True where position t may read position i, that is where i <= t."""
    raise NotImplementedError("TODO 2")

In [ ]:
#@title Solution 2 — try it yourself first { display-mode: "form" }
@workshop.solution(2)
def causal_mask(T):
    """(T, T) bool tensor, True where position t may read position i, that is where i <= t."""
    return torch.tril(torch.ones(T, T, dtype=torch.bool))  # row t: True in columns 0..t

<details>
<summary>Why this solution works</summary>

`torch.tril` keeps the lower triangle, diagonal included: row $t$ is `True` in columns $0, \dots, t$. Passed to Exercise 1's function, every `False` entry becomes $-\infty$, so $\alpha_{t,i} = 0$ exactly for $i > t$ and $A$ is lower triangular (briefing eq. `mask`). Row $t$ always keeps its diagonal entry, so no row is empty and the softmax is always defined.

</details>

In [ ]:
# Checkpoint 2: the leak test on one attention step.
workshop.checkpoint(2)
_M = causal_mask(4)
assert isinstance(_M, torch.Tensor) and _M.dtype == torch.bool, "return a bool tensor"
assert _M.tolist() == [[True, False, False, False], [True, True, False, False],
                       [True, True, True, False], [True, True, True, True]], (
    f"causal_mask(4) must be True on and below the diagonal; got {_M.int().tolist()}: "
    "use the lower triangle, diagonal included")

_g = torch.Generator().manual_seed(1)
T, d, d_k, t0 = 10, 16, 8, 6
W_Q, W_K, W_V = (torch.randn(d, d_k, generator=_g) / math.sqrt(d) for _ in range(3))


def _attend(H):  # one masked self-attention step on one sequence H: (T, d)
    return scaled_dot_product_attention(H @ W_Q, H @ W_K, H @ W_V, causal_mask(T))


H = torch.randn(T, d, generator=_g)
V_bar, A = _attend(H)
assert (A.triu(diagonal=1) == 0).all(), "every weight above the diagonal must be exactly 0"
assert torch.allclose(A.sum(-1), torch.ones(T), atol=1e-6)

H_changed = H.clone()
H_changed[t0] = torch.randn(d, generator=_g)  # change the input at position t0 only
V_bar_changed, _ = _attend(H_changed)
change_before = (V_bar_changed[:t0] - V_bar[:t0]).abs().max().item()
change_from_t0 = (V_bar_changed[t0:] - V_bar[t0:]).abs().max().item()
print(f"max |change| at positions < {t0}:  {change_before:.2e}")
print(f"max |change| at positions >= {t0}: {change_from_t0:.2e}")
assert change_before <= 1e-6, "an earlier output changed: the layer reads ahead"
assert change_from_t0 > 1e-3, "nothing changed from t0 on: is the mask the wrong way round?"
print("Checkpoint 2 passed: no position reads ahead")

**Explain.** Positions before $t_0$ do not change at all, not even in the last decimal: their weights on position $t_0$ are exactly zero, so the new value enters their sums multiplied by 0. Positions from $t_0$ on read the changed input and do change. The same test on the whole model comes at the end of Exercise 3, once the model is complete, and again after training.

## Exercise 3 · Positions (6 minutes)

Self-attention treats what it reads as a set. Position $t$'s output is a weighted sum over the positions it reads, and each term depends only on contents, so shuffling the earlier positions only reorders the sum. The fix (briefing eq. `pos`) adds a vector that depends only on the position:

$$
h_t^{(0)} = e_{w_t} + p_t .
$$

Here $p_t$ is a row of a second embedding table, indexed by position $0, \dots, T_{\max} - 1$ and learned with everything else.

**Predict.** The cell below feeds one attention layer with token embeddings only, no positions. It shuffles every token except the last, then compares the output at the last position before and after. What difference do you expect?

In [ ]:
# Provided: one attention layer, no positions, shuffle the tokens before the last position.
torch.manual_seed(SEED)
d_demo, T_demo = 32, 12
tok_demo = nn.Embedding(VOCAB_SIZE, d_demo)  # token embeddings e_w
pos_demo = nn.Embedding(T_demo, d_demo)  # a position table p_t, used after your Exercise 3
layer_demo = MultiHeadAttention(d_demo, n_h=4).eval()  # ONE layer: the test holds only for one
mask_demo = causal_mask(T_demo)

_g = torch.Generator().manual_seed(2)
idx_demo = torch.randint(0, VOCAB_SIZE, (1, T_demo), generator=_g)
perm = torch.randperm(T_demo - 1, generator=_g)  # a shuffle of positions 0 .. T-2
idx_shuffled = torch.cat([idx_demo[:, perm], idx_demo[:, -1:]], dim=1)  # last token stays last
assert not torch.equal(idx_demo, idx_shuffled)


def last_position_change(h0, h0_shuffled):
    """Max |difference| of the attention layer's output at the last position."""
    with torch.no_grad():
        out = layer_demo(h0, mask_demo)[0][0, -1]  # (d,)
        out_shuffled = layer_demo(h0_shuffled, mask_demo)[0][0, -1]
    return (out - out_shuffled).abs().max().item()


change_no_positions = last_position_change(tok_demo(idx_demo), tok_demo(idx_shuffled))
print("tokens:  ", repr("".join(chars[i] for i in idx_demo[0].tolist())))
print("shuffled:", repr("".join(chars[i] for i in idx_shuffled[0].tolist())))
print(f"Without positions: max |change| at the last position = {change_no_positions:.2e}")
assert change_no_positions < 1e-5

The last position's output did not change (up to rounding): the layer cannot tell the two orders apart.

**Task.** Complete `embed_with_positions(idx, tok_emb, pos_emb)`, the input step of the model. `idx` has shape `(B, T)`. Look up the token embeddings, look up the position embeddings for positions $0, \dots, T - 1$ (create them with `torch.arange`, on the same device as `idx`), and add. Broadcasting adds the same `(T, d)` position table to every sequence in the batch.

In [ ]:
# TODO 3: complete this function.
def embed_with_positions(idx, tok_emb, pos_emb):
    """h0 = e_{w_t} + p_t. idx: (B, T) IDs; tok_emb, pos_emb: nn.Embedding. Returns (B, T, d)."""
    raise NotImplementedError("TODO 3")

In [ ]:
#@title Solution 3 — try it yourself first { display-mode: "form" }
@workshop.solution(3)
def embed_with_positions(idx, tok_emb, pos_emb):
    """h0 = e_{w_t} + p_t. idx: (B, T) IDs; tok_emb, pos_emb: nn.Embedding. Returns (B, T, d)."""
    T = idx.size(1)
    positions = torch.arange(T, device=idx.device)  # 0, 1, ..., T-1
    return tok_emb(idx) + pos_emb(positions)  # (B, T, d) + (T, d), broadcast over the batch

<details>
<summary>Why this solution works</summary>

`tok_emb(idx)` has shape `(B, T, d)`: row $t$ is $e_{w_t}$. `pos_emb(positions)` has shape `(T, d)`: row $t$ is $p_t$. Adding them broadcasts the position table over the batch, which is briefing eq. `pos` for every sequence at once. The same character now has a different vector at each position, so queries and keys, and hence the attention weights, can depend on where a token is. Because the table has $T_{\max}$ rows, the model cannot accept a longer input: `GPT.forward` asserts it.

</details>

In [ ]:
# Checkpoint 3
workshop.checkpoint(3)
h0 = embed_with_positions(idx_demo, tok_emb=tok_demo, pos_emb=pos_demo)
assert h0.shape == (1, T_demo, d_demo), f"h0 has shape {tuple(h0.shape)}"
assert torch.allclose(h0[0, 5], tok_demo.weight[idx_demo[0, 5]] + pos_demo.weight[5]), (
    f"h0 at position 5 must be e_w + p_5; it differs by up to "
    f"{(h0[0, 5] - tok_demo.weight[idx_demo[0, 5]] - pos_demo.weight[5]).abs().max().item():.3g}: "
    "look up the position table at 0, ..., T - 1 and add it to the token embeddings")
change_with_positions = last_position_change(h0, embed_with_positions(idx_shuffled, tok_demo, pos_demo))
print(f"Without positions: max |change| at the last position = {change_no_positions:.2e}")
print(f"With positions:    max |change| at the last position = {change_with_positions:.2e}")
assert change_with_positions > 1e-3, "with positions the shuffle must change the output"
print("Checkpoint 3 passed: with positions the layer sees the order")

**Explain.** With positions the shuffle now changes the last output: the same characters at different positions give different queries and keys. The shuffle test is only valid for a **single** layer. In a stack of causal layers, deeper layers can partly infer position from the mask itself (position $t$ reads $t + 1$ positions), so do not use this test on the full model.

**Run: the leak test on the whole model.** The model is now complete. Before training it, run the leak test of Exercise 2 on its logits. Dropout is off (`eval()`), and no layer mixes examples or positions except the masked attention, so the test must hold exactly.

In [ ]:
# Provided: the leak test on the whole untrained model, on its logits.
workshop.checkpoint(label="leak test")
def model_leak_test(model, t0=50, T=None, seed=3):
    """Change the input at t0; return max |change| of the logits before t0 and from t0 on."""
    model.eval()
    T = T or model.T_max
    g = torch.Generator().manual_seed(seed)
    idx = torch.randint(0, VOCAB_SIZE, (2, T), generator=g).to(DEVICE)
    idx_changed = idx.clone()
    idx_changed[:, t0] = (idx[:, t0] + 1) % VOCAB_SIZE  # a different character at t0
    with torch.no_grad():
        logits, logits_changed = model(idx), model(idx_changed)  # (2, T, |V|) each
    before = (logits_changed[:, :t0] - logits[:, :t0]).abs().max().item()
    after = (logits_changed[:, t0:] - logits[:, t0:]).abs().max().item()
    return before, after


CONFIG = {"d": 128, "n_h": 4, "n_l": 4, "T_max": 128, "d_ff": 512, "dropout": 0.1}
torch.manual_seed(SEED)
_untrained = GPT(VOCAB_SIZE, **CONFIG).to(DEVICE)
_before, _after = model_leak_test(_untrained)
print(f"Untrained model: max |change| of logits before t0 = {_before:.2e}, from t0 on = {_after:.2e}")
assert _before <= 1e-5 and _after > 0, "the model reads ahead"
print("Checkpoint 3b passed: the whole model does not read ahead")

# Part B · Training and Comparing

## Exercise 4 · Train the Mini-GPT and Compare It With the LSTM (14 minutes)

Nothing new to write: start training, then make your predictions while it runs.

**The models.** Each row of the results table is trained or counted on the same 1,000,000 training characters and scored on the same test characters.

| Model | What it is |
|---|---|
| Character trigram and 5-gram | Lab 1's add-k n-gram, recomputed here from counts; $k$ as chosen on val in Lab 1 |
| LSTM, measured in Lab 3 | Lab 3's own result, quoted from `data/baselines.json` (`lab03.lstm_lm`): embedding $d = 64$, one `nn.LSTM` layer with $d_h = 256$, Adam at a constant learning rate 0.002, batch 64, 2,000 steps |
| LSTM, trained here | The same architecture (with PyTorch's default initialization; Lab 3 also sets the forget-gate bias to 1), trained here with the GPT's loop (warm-up and cosine decay, learning rate chosen on val) and the GPT's batch size and number of steps, so the two models are compared at one budget |
| Mini-GPT | The model above: $d = 128$, $n_h = 4$ ($d_k = 32$), $n_\ell = 4$, $T_{\max} = 128$, $d_{\mathrm{ff}} = 512$, dropout 0.1 |

**The training loop** (provided) samples random windows of 128 characters, uses AdamW with a short linear warm-up and a cosine decay of the learning rate, and clips the gradient norm at 1 (Module 3).

**Predict.** Rank the trigram, the 5-gram, the LSTM and the mini-GPT by test perplexity. Which pair do you expect to be closest? The briefing deliberately makes no promise about GPT against LSTM at this size.

In [ ]:
TRAIN = {  # step counts set by the compute budget; see the notes under Checkpoint 4b
    False: {"batch_size": 64, "gpt_steps": 3000, "lstm_steps": 3000},
    True: {"batch_size": 16, "gpt_steps": 1500, "lstm_steps": 1500},  # QUICK
}[QUICK]
# Peak learning rates, chosen on val with the QUICK settings from {1, 2, 3, 5, 8} x 1e-3 (GPT) and
# {1, 2, 4, 8, 16} x 1e-3 (LSTM); ties within 0.001 nats go to the smaller rate.
GPT_LR, LSTM_LR = 5e-3, 8e-3
LSTM_CONFIG = {"d": 64, "d_h": 256}  # Lab 3's LSTM


class LSTMLM(nn.Module):
    """Lab 3's character LSTM language model, restated: embed, one LSTM layer, output layer."""

    def __init__(self, vocab_size, d=64, d_h=256):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d)
        self.lstm = nn.LSTM(d, d_h, batch_first=True)
        self.W_o = nn.Linear(d_h, vocab_size)

    def forward(self, idx, state=None, return_state=False):
        out, state = self.lstm(self.emb(idx), state)  # out: (B, T, d_h)
        logits = self.W_o(out)  # (B, T, |V|)
        return (logits, state) if return_state else logits


_g = torch.Generator().manual_seed(1234)  # fixed validation windows for the progress printout
VAL_PROBE = [get_batch(ids["val"], 32, CONFIG["T_max"], _g) for _ in range(4)]


@torch.no_grad()
def quick_val_loss(model):
    """Mean loss on 4 x 32 fixed validation windows (a progress estimate, not the reported metric)."""
    model.eval()
    loss = sum(lm_loss_and_ppl(model(x), y)[0].item() for x, y in VAL_PROBE) / len(VAL_PROBE)
    model.train()
    return loss


def train_lm(model, steps, lr, batch_size, T=128, clip=1.0, seed=SEED, log_every=None):
    """Provided training loop. Returns (history, training seconds excluding the progress estimates)."""
    gen = torch.Generator().manual_seed(seed)  # the order of the training windows
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.99), weight_decay=0.01)
    warmup = max(1, min(100, steps // 10))

    def lr_factor(step):  # linear warm-up, then cosine decay to 10% of the peak
        if step < warmup:
            return (step + 1) / warmup
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (step - warmup) / (steps - warmup)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
    log_every = log_every or max(1, steps // 6)
    history, eval_seconds, started = [], 0.0, time.time()
    model.train()
    for step in range(1, steps + 1):
        x, y = get_batch(ids["train"], batch_size, T, gen)
        loss = lm_loss_and_ppl(model(x), y)[0]
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()
        sched.step()
        if step % log_every == 0 or step == steps:
            eval_started = time.time()
            val = quick_val_loss(model)
            eval_seconds += time.time() - eval_started
            history.append((step, loss.item(), val))
            print(f"  step {step:>5}  train loss {loss.item():.3f}  val loss (estimate) {val:.3f}"
                  f"  {time.time() - started:6.0f} s")
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    model.eval()
    return history, time.time() - started - eval_seconds


torch.manual_seed(SEED)
gpt = GPT(VOCAB_SIZE, **CONFIG).to(DEVICE)
torch.manual_seed(SEED)
lstm = LSTMLM(VOCAB_SIZE, **LSTM_CONFIG).to(DEVICE)
PARAMS = {"gpt": count_parameters(gpt), "lstm": count_parameters(lstm)}
print(f"Mini-GPT: {PARAMS['gpt']:,} parameters · LSTM: {PARAMS['lstm']:,} parameters")
print(f"QUICK = {QUICK}: batch {TRAIN['batch_size']}, GPT {TRAIN['gpt_steps']} steps, "
      f"LSTM {TRAIN['lstm_steps']} steps, windows of {CONFIG['T_max']} characters")

untrained_loss = quick_val_loss(gpt)
print(f"\nUntrained GPT: val loss {untrained_loss:.4f} nats; ln |V| = {math.log(VOCAB_SIZE):.4f}")

In [ ]:
# Checkpoint 4a: an untrained model with small random weights is close to uniform.
workshop.checkpoint(label="4a")
assert abs(untrained_loss - math.log(VOCAB_SIZE)) < 0.05, untrained_loss
print("Checkpoint 4a passed: the untrained GPT predicts close to uniformly")
TIMES["setup and Exercises 1-3"] = time.time() - LAB_START

In [ ]:
print("Training the mini-GPT ...")
gpt_history, gpt_seconds = train_lm(gpt, TRAIN["gpt_steps"], GPT_LR, TRAIN["batch_size"], CONFIG["T_max"])
print(f"Mini-GPT trained in {gpt_seconds:.0f} s")
TIMES["train mini-GPT"] = gpt_seconds

In [ ]:
print("Training the LSTM ...")
lstm_history, lstm_seconds = train_lm(lstm, TRAIN["lstm_steps"], LSTM_LR, TRAIN["batch_size"], CONFIG["T_max"])
print(f"LSTM trained in {lstm_seconds:.0f} s")
TIMES["train LSTM"] = lstm_seconds

**Run: score every model on the same characters.** Three rules make the comparison fair:

1. **Same characters.** Every model scores all 60,394 characters of the test split, each exactly once. The checks below assert it.
2. **Same metric.** Lab 3's `lm_loss_and_ppl`: mean negative log-likelihood in nats per character, its exponential (perplexity per character) and bits per character.
3. **Context from the text before.** The splits are contiguous, so the first test characters are predicted from the end of the validation text, as in Lab 1.

The models are scored on the same characters **but with different context**, because they read context differently:

- The **n-gram** sees the previous $n - 1$ characters.
- The **LSTM** carries its state forward: it reads the 1,000 characters before the split to warm up, then the whole split in chunks of 1,000 with the state carried across them, so no character is scored from a cold state. This is Lab 3's evaluation routine.
- The **GPT** sees at most $T_{\max} = 128$ characters. Scoring non-overlapping windows of 128 would give the first characters of each window almost no context and understate the model. So the windows overlap: each window of 128 is moved forward by 64, and only its second half is scored. Every character then has between 65 and 128 characters of context.

In [ ]:
def overlapping_windows(n_stream, first, T_max):
    """Windows over a stream that score each target index >= first exactly once.

    Yields (s, lo, hi): the window's inputs are stream[s : s + T_max]; window positions lo..hi-1
    are scored, and position j predicts stream[s + 1 + j]. Every scored character has at least
    T_max // 2 + 1 characters of context.
    """
    half = T_max // 2
    target = first  # the next index of the stream to score
    while target < n_stream:
        s = min(target - 1 - half, n_stream - 1 - T_max)  # the last window ends at the stream's end
        lo = target - 1 - s
        yield s, lo, T_max
        target = s + 1 + T_max


@torch.no_grad()
def evaluate_gpt(model, split, batch_size=64):
    """Score every character of a split once with overlapping windows. Returns a dict of metrics."""
    model.eval()
    T_max = model.T_max
    first = SPLIT_START[split]
    stream = all_ids[first - T_max : first + len(ids[split])]  # T_max characters of context first
    windows = list(overlapping_windows(len(stream), T_max, T_max))
    logits_kept, targets_kept = [], []
    for b in range(0, len(windows), batch_size):
        chunk = windows[b : b + batch_size]
        x = torch.stack([stream[s : s + T_max] for s, _, _ in chunk]).to(DEVICE)  # (b, T_max)
        logits = model(x)  # (b, T_max, |V|)
        for row, (s, lo, hi) in enumerate(chunk):
            logits_kept.append(logits[row, lo:hi].cpu())
            targets_kept.append(stream[s + 1 + lo : s + 1 + hi])
    logits_kept, targets_kept = torch.cat(logits_kept), torch.cat(targets_kept)
    assert torch.equal(targets_kept, ids[split]), "every character of the split, once, in order"
    nll, ppl, bpc = lm_loss_and_ppl(logits_kept, targets_kept)
    return {"nats": nll.item(), "ppl": ppl.item(), "bpc": bpc.item(), "scored": len(targets_kept)}


@torch.no_grad()
def evaluate_lstm(model, split, warmup=1000, chunk=1000):
    """Score every character of a split once, carrying the LSTM state through the whole stream."""
    model.eval()
    first = SPLIT_START[split]
    stream = all_ids[first - warmup : first + len(ids[split])]  # warm-up text, then the split
    inputs, targets = stream[:-1], stream[1:]
    state, logits_kept = None, []
    for b in range(0, len(inputs), chunk):  # contiguous chunks, state carried across them
        logits, state = model(inputs[b : b + chunk][None].to(DEVICE), state, return_state=True)
        logits_kept.append(logits[0].cpu())
    logits_kept = torch.cat(logits_kept)[warmup - 1 :]  # keep the predictions of the split only
    targets_kept = targets[warmup - 1 :]
    assert torch.equal(targets_kept, ids[split]), "every character of the split, once, in order"
    nll, ppl, bpc = lm_loss_and_ppl(logits_kept, targets_kept)
    return {"nats": nll.item(), "ppl": ppl.item(), "bpc": bpc.item(), "scored": len(targets_kept)}


NGRAM_BASE = VOCAB_SIZE + 1  # character IDs 0..64, plus the start ID 65 that pads the start of train


def ngram_codes(seq, n):
    """An integer code for every window of n IDs in seq (base NGRAM_BASE)."""
    codes = np.zeros(len(seq) - n + 1, dtype=np.int64)
    for j in range(n):
        codes = codes * NGRAM_BASE + seq[j : len(seq) - n + 1 + j]
    return codes


def char_ngram_eval(n, k, split, counts):
    """Lab 1's add-k character n-gram, vectorized. Returns a dict of metrics on `split`."""
    gram_keys, gram_counts, ctx_keys, ctx_counts = counts
    first = SPLIT_START[split]
    seq = all_ids[first - (n - 1) : first + len(ids[split])].numpy()  # n - 1 characters of history
    query = ngram_codes(seq, n)  # one code per scored character

    def lookup(keys, values, q):
        pos = np.minimum(np.searchsorted(keys, q), len(keys) - 1)
        return np.where(keys[pos] == q, values[pos], 0)

    p = (lookup(gram_keys, gram_counts, query) + k) / (
        lookup(ctx_keys, ctx_counts, query // NGRAM_BASE) + k * VOCAB_SIZE
    )
    nats = float(-np.log(p).mean())
    return {"nats": nats, "ppl": math.exp(nats), "bpc": nats / math.log(2), "scored": len(query)}


def char_ngram_counts(n):
    """Counts from train, padded at its start with n - 1 start IDs, as in Lab 1."""
    train = np.concatenate([np.full(n - 1, VOCAB_SIZE), ids["train"].numpy()])
    gram_keys, gram_counts = np.unique(ngram_codes(train, n), return_counts=True)  # c(context, w)
    # c(context) = sum over w of c(context, w); a context's code is its n-gram's code without w
    ctx_keys, inverse = np.unique(gram_keys // NGRAM_BASE, return_inverse=True)
    ctx_counts = np.bincount(inverse, weights=gram_counts).astype(np.int64)
    return gram_keys, gram_counts, ctx_keys, ctx_counts

In [ ]:
started = time.time()
NGRAM_K = {3: 0.1, 5: 0.01}  # k chosen on val in Lab 1 (data/baselines.json)
ngram = {}
for n, k in NGRAM_K.items():
    counts = char_ngram_counts(n)
    ngram[n] = {split: char_ngram_eval(n, k, split, counts) for split in ("val", "test")}
ngram_seconds = time.time() - started
TIMES["n-gram baselines"] = ngram_seconds

started = time.time()
results = {
    "gpt": {split: evaluate_gpt(gpt, split) for split in ("val", "test")},
    "lstm": {split: evaluate_lstm(lstm, split) for split in ("val", "test")},
}
TIMES["evaluation"] = time.time() - started

# Measured in Lab 3 (data/baselines.json, lab03.lstm_lm): batch 64, 2,000 steps, CPU, seed 0.
LAB3_LSTM = {"val": {"nats": 1.5316}, "test": {"nats": 1.6114, "ppl": 5.0098, "bpc": 2.3247, "scored": 60_394}}

rows = [
    ("character trigram (Lab 1's method)", ngram[3], "-", "counts", f"{ngram_seconds:.0f}"),
    ("character 5-gram (Lab 1's method)", ngram[5], "-", "counts", f"{ngram_seconds:.0f}"),
    ("LSTM, measured in Lab 3", LAB3_LSTM, "-", "2000", "Lab 3"),
    ("LSTM, trained here", results["lstm"], f"{PARAMS['lstm']:,}", TRAIN["lstm_steps"], f"{lstm_seconds:.0f}"),
    ("mini-GPT", results["gpt"], f"{PARAMS['gpt']:,}", TRAIN["gpt_steps"], f"{gpt_seconds:.0f}"),
]
print(f"Tiny Shakespeare, character level, |V| = 65 · QUICK = {QUICK} · device {DEVICE}")
print(f"{'model':<36} {'val nats':>9} {'test nats':>10} {'test PPL':>9} {'test BPC':>9}"
      f" {'params':>9} {'steps':>7} {'train s':>8}")
for name, r, params, steps, seconds in rows:
    print(f"{name:<36} {r['val']['nats']:>9.4f} {r['test']['nats']:>10.4f} {r['test']['ppl']:>9.4f}"
          f" {r['test']['bpc']:>9.4f} {params:>9} {steps:>7} {seconds:>8}")
print(f"\nScored test characters: n-gram {ngram[3]['test']['scored']:,} · LSTM "
      f"{results['lstm']['test']['scored']:,} · GPT {results['gpt']['test']['scored']:,}")
print("Training seconds count the training loop only, on this device; the n-gram rows give the counting time.\n"
      "The Lab 3 row is quoted, not recomputed: its numbers were measured on a CPU in Lab 3.")

**Explain.** Compare the table with your ranking.

- Both neural models are well below the trigram; Checkpoint 4b asserts it for the GPT. Compare each with the 5-gram, Lab 1's best: the neural models see far more context than $n - 1$ characters and never meet an "unseen context", but how much that buys depends on how long they were trained.
- The two LSTM rows are the same architecture with different training. The one trained here uses the GPT's loop, batch size and number of steps, with its own learning rate chosen on val, so it is the fair partner for the GPT. Lab 3's row shows what Lab 3's simpler settings gave.
- Read the GPT-against-LSTM comparison **at the stated budget**: the parameter count, the number of steps and the training time are printed beside each test number. The models have different sizes and a different cost per step, and a different budget could reorder them. The briefing's point about the transformer is parallel training and short paths, which pay off at scale; at this size either model can win.
- The models see different context at test time (above), so this compares the models as they would be used, not their architectures in isolation.
- With `QUICK = True`, both models trained here are undertrained and their numbers are not the lab's reference numbers.

In [ ]:
# Checkpoint 4b
workshop.checkpoint(label="4b")
LAB1_NATS = {3: 2.0859, 5: 1.8326}  # test nats per character measured in Lab 1 (data/baselines.json)
for n, expected in LAB1_NATS.items():
    assert abs(ngram[n]["test"]["nats"] - expected) < 2e-3, (n, ngram[n]["test"]["nats"], expected)
assert ngram[3]["test"]["scored"] == results["lstm"]["test"]["scored"] == results["gpt"]["test"]["scored"] == 60_394
GPT_MARGIN = 0.15  # nats; see the note on thresholds below
assert results["gpt"]["test"]["nats"] < ngram[3]["test"]["nats"] - GPT_MARGIN, results["gpt"]["test"]
_before, _after = model_leak_test(gpt)
print(f"Trained GPT leak test: max |change| of logits before t0 = {_before:.2e}, from t0 on = {_after:.2e}")
assert _before <= 1e-5 and _after > 1e-3
print("Checkpoint 4b passed: Lab 1's n-gram values reproduced, the same 60,394 test characters scored "
      "by every model, the GPT beats the trigram, and the trained GPT does not read ahead")

<details>
<summary>How the thresholds in this lab were set</summary>

- **Checkpoint 4b, GPT against the trigram.** With `QUICK = True` the GPT's test cross-entropy was 1.770, 1.791 and 1.794 nats at seeds 0, 1 and 2, that is 0.29 to 0.32 nats below the trigram's 2.086. The margin of 0.15 nats leaves room for run-to-run variation; the GPU settings train longer.
- **Checkpoint 4a.** The untrained GPT is built with a fixed seed; its loss was 4.191 nats against $\ln 65 = 4.174$.
- **Learning rates** were chosen on val with the `QUICK` settings, from 1, 2, 3, 5 and 8 times $10^{-3}$ for the GPT and 1, 2, 4, 8 and 16 times $10^{-3}$ for the LSTM, and are used unchanged on the GPU. They were not tuned at batch 64.
- There is deliberately no assertion on GPT against LSTM. With `QUICK = True` the LSTM trained here was better at all three seeds (1.562 to 1.582 nats).

</details>

**Run: samples.** The same prompt, the same temperatures as Lab 3 ($\tau = 0.5, 1.0, 1.5$) and greedy decoding, for both models. The GPT only ever sees the last 128 characters of what it has written; the LSTM carries its state through the whole sample.

**Predict.** At which temperature will the two models be hardest to tell apart?

In [ ]:
def next_characters(z, taus, generator):
    """One next character per row of z: (n, |V|) logits. tau = 0 means greedy (arg max)."""
    return [int(z[r].argmax()) if tau == 0 else int(sample_with_temperature(z[r], tau, generator))
            for r, tau in enumerate(taus)]


@torch.no_grad()
def generate_gpt(model, prompt, length, taus, seed):
    """One sample of `length` characters after `prompt` per temperature, generated as a batch."""
    model.eval()
    generator = torch.Generator().manual_seed(seed)
    idx = torch.tensor([[char_to_id[ch] for ch in prompt]] * len(taus), device=DEVICE)  # (n, len)
    for _ in range(length):
        z = model(idx[:, -model.T_max :])[:, -1].cpu()  # next-character logits: (n, |V|)
        nxt = torch.tensor(next_characters(z, taus, generator), device=DEVICE)[:, None]
        idx = torch.cat([idx, nxt], dim=1)  # the GPT re-reads the last T_max characters
    return ["".join(chars[i] for i in row[len(prompt) :]) for row in idx.tolist()]


@torch.no_grad()
def generate_lstm(model, prompt, length, taus, seed):
    """One sample of `length` characters after `prompt` per temperature, generated as a batch."""
    model.eval()
    generator = torch.Generator().manual_seed(seed)
    x = torch.tensor([[char_to_id[ch] for ch in prompt]] * len(taus), device=DEVICE)
    logits, state = model(x, return_state=True)
    out = []
    for _ in range(length):
        nxt = next_characters(logits[:, -1].cpu(), taus, generator)
        out.append(nxt)
        logits, state = model(torch.tensor(nxt, device=DEVICE)[:, None], state, return_state=True)
    return ["".join(chars[step[r]] for step in out) for r in range(len(taus))]  # the state carries on


started = time.time()
PROMPT, SAMPLE_LENGTH, TAUS = "ROMEO:\n", 300, (0.5, 1.0, 1.5, 0)  # 0 = greedy
samples = {}
for name, generate, model in [("mini-GPT", generate_gpt, gpt), ("LSTM", generate_lstm, lstm)]:
    for tau, text in zip(TAUS, generate(model, PROMPT, SAMPLE_LENGTH, TAUS, SEED)):
        samples[name, tau] = text
for tau in TAUS:
    for name in ("mini-GPT", "LSTM"):
        label = "greedy" if tau == 0 else f"tau = {tau}"
        print(f"----- {name}, {label} -----\n{PROMPT}{samples[name, tau]}\n")
TIMES["samples"] = time.time() - started

**Explain.** At $\tau = 0.5$ both models produce plausible names, line breaks and common words, and repeat themselves; at $\tau = 1.5$ both fall apart into invented words. Greedy decoding usually locks into a loop within a few lines, for both architectures: that is a property of arg max decoding, not of the model (Module 3). Differences between the two models are easiest to see at $\tau = 1.0$, where each samples from its own distribution.

## Exercise 5 · Looking at the Heads (7 minutes)

Each layer has $n_h$ weight matrices $A^{(r)}$, one per head. A simple summary of a head is how far back it reads. The **mean attention distance** of one head on one sequence is

$$
\bar{\delta} = \frac{1}{T} \sum_{t} \sum_{i \le t} \alpha_{t,i}\, (t - i),
$$

the average over positions $t$ of the expected offset of the positions that $t$ reads. A head that reads only the current position has $\bar{\delta} = 0$; one that reads only the previous position has $\bar{\delta} \approx 1$.

**Predict.** Will the heads of the first layer read further back or less far back than the heads of the last layer? Will all heads of a layer behave alike?

**Task.** Complete `mean_attention_distance(A)` for `A` of shape `(..., T, T)`; return a tensor of shape `(...)`. Build the matrix of offsets $t - i$ with `torch.arange`; no loops are needed.

<details>
<summary>Hint for TODO 5</summary>

The offset is $t - i$, the query's row minus the key's column, so it is never negative where the causal mask allows reading; $i - t$ gives negative distances. Sum the weighted offsets over the key axis, the last one, to get one expected offset per row, then average over the $T$ rows. The identity matrix must give 0 and a previous-token head $(T - 1)/T$.

</details>

In [ ]:
# TODO 5: complete this function.
def mean_attention_distance(A):
    """Mean over t of sum_i A[..., t, i] * (t - i). A: (..., T, T) weights. Returns shape (...)."""
    raise NotImplementedError("TODO 5")

In [ ]:
#@title Solution 5 — try it yourself first { display-mode: "form" }
@workshop.solution(5)
def mean_attention_distance(A):
    """Mean over t of sum_i A[..., t, i] * (t - i). A: (..., T, T) weights. Returns shape (...)."""
    T = A.size(-1)
    positions = torch.arange(T, device=A.device)
    offsets = (positions[:, None] - positions[None, :]).to(A.dtype)  # (T, T), entry (t, i) is t - i
    return (A * offsets).sum(dim=-1).mean(dim=-1)  # expected offset per row t, then the mean over t

<details>
<summary>Why this solution works</summary>

`offsets[t, i]` is $t - i$. Multiplying elementwise by $A$ and summing over the last dimension gives, for each row $t$, $\sum_i \alpha_{t,i}(t - i)$: the expected offset under row $t$'s distribution. The mean over the rows is $\bar{\delta}$. The entries above the diagonal have negative offsets, but their weights are exactly 0 under the causal mask, so they add nothing. Broadcasting handles any leading dimensions, so one call summarizes every head of every sequence in a batch.

</details>

In [ ]:
# Checkpoint 5
workshop.checkpoint(5)
_T = 8
_identity = torch.eye(_T)
_previous = torch.zeros(_T, _T)
_previous[0, 0] = 1.0  # row 0 can only read itself
_previous[torch.arange(1, _T), torch.arange(_T - 1)] = 1.0  # every other row reads t - 1
_uniform = causal_mask(_T).float() / torch.arange(1, _T + 1)[:, None]  # row t: 1/(t+1) on 0..t
assert abs(mean_attention_distance(_identity).item()) < 1e-6, (
    f"reading only yourself gives distance 0, got {mean_attention_distance(_identity).item()}")
assert abs(mean_attention_distance(_previous).item() - (_T - 1) / _T) < 1e-6, (
    f"previous-token head: 1 on every row except row 0, so (T - 1) / T = {(_T - 1) / _T}; "
    f"got {mean_attention_distance(_previous).item()}: the offset is t - i (row minus column)"
)
assert abs(mean_attention_distance(_uniform).item() - (_T - 1) / 4) < 1e-6, (
    f"uniform head: t / 2 per row, so (T - 1) / 4 = {(_T - 1) / 4} on average; "
    f"got {mean_attention_distance(_uniform).item()}: weight every offset by its attention weight")
assert mean_attention_distance(torch.stack([_identity, _previous]).expand(3, 2, _T, _T)).shape == (3, 2), (
    f"one distance per leading index: expected shape (3, 2), got "
    f"{tuple(mean_attention_distance(torch.stack([_identity, _previous]).expand(3, 2, _T, _T)).shape)}: "
    "reduce over the last two axes only")
print("Checkpoint 5 passed")

**Run.** The cell below computes $\bar{\delta}$ for every head of the trained GPT, averaged over 64 validation windows of 128 characters. For comparison, a head that spread its weight uniformly over everything it may read would have $\bar{\delta} = (T - 1)/4 \approx 32$. Then it draws every head of the first and of the last layer on a short passage from the test split.

In [ ]:
started = time.time()
_g = torch.Generator().manual_seed(5)
x_probe, _ = get_batch(ids["val"], 64, CONFIG["T_max"], _g)
with torch.no_grad():
    _, attention = gpt(x_probe, return_attention=True)  # n_l tensors of shape (64, n_h, 128, 128)
distance = torch.stack([mean_attention_distance(A).mean(dim=0) for A in attention]).cpu()  # (n_l, n_h)

print("Mean attention distance in characters (rows: layers, columns: heads)")
print("         " + "".join(f"head {r + 1:<4}" for r in range(CONFIG["n_h"])))
for layer, row in enumerate(distance.tolist()):
    print(f"layer {layer + 1}  " + "".join(f"{v:>8.2f}  " for v in row))
print(f"A uniform head over all allowed positions would give {(CONFIG['T_max'] - 1) / 4:.2f}")


def plot_heads(A_layer, text, title):
    """Draw the n_h weight matrices of one layer for one passage. A_layer: (n_h, T, T)."""
    n_h, T, _ = A_layer.shape
    assert (A_layer.triu(diagonal=1) == 0).all(), "a plotted A must be lower triangular"
    assert torch.allclose(A_layer.sum(-1), torch.ones(n_h, T), atol=1e-5), "rows must sum to 1"
    labels = [repr(ch)[1:-1] if ch != " " else "␣" for ch in text]
    fig, axes = plt.subplots(1, n_h, figsize=(4 * n_h, 4.2))
    for r, ax in enumerate(axes):
        ax.imshow(A_layer[r].numpy(), cmap="Blues", vmin=0, vmax=1)
        ax.set_title(f"head {r + 1}")
        ax.set_xticks(range(T), labels, fontsize=6, rotation=90)
        ax.set_yticks(range(T), labels, fontsize=6)
        ax.set_xlabel("position i (is read)")
    axes[0].set_ylabel("position t (asks)")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


PASSAGE = "GRUMIO:\nI confess two sleeves.\n"  # from the test split
assert PASSAGE in corpus["test"]
with torch.no_grad():
    _, passage_attention = gpt(torch.tensor([[char_to_id[ch] for ch in PASSAGE]], device=DEVICE),
                               return_attention=True)
passage_attention = [A[0].cpu() for A in passage_attention]  # n_l tensors of shape (n_h, T, T)
plot_heads(passage_attention[0], PASSAGE, "Layer 1: every head on one passage")
plot_heads(passage_attention[-1], PASSAGE, f"Layer {CONFIG['n_l']}: every head on one passage")
TIMES["Exercise 5"] = time.time() - started

**Explain.** Write two sentences in the cell below (double-click to edit):

1. Which heads read mostly the last one or two characters, and which spread their weight further back? Compare with your prediction.
2. Find one head whose pattern you can describe in words on the passage (for example: "reads the previous character", "reads the start of the line").

*Your answer:*

**A caution, from the briefing.** An attention map is not an explanation of the prediction. There are $n_h \times n_\ell = 16$ maps, and from layer 2 on the vectors being read are already mixtures of many characters, so "head 3 reads position 7" does not mean it reads character 7. Mean distance is a summary of where the weight goes, not of what the head does. Treat any story you tell about a head as a hypothesis to test.

In [ ]:
# The baseline card and the attention data: every number measured in this run.
card = {
    "quick": QUICK,
    "device": DEVICE if DEVICE == "cpu" else torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "config": {"gpt": CONFIG, "lstm": LSTM_CONFIG, "train": TRAIN, "gpt_lr": GPT_LR, "lstm_lr": LSTM_LR},
    "parameters": PARAMS,
    "train_seconds": {"gpt": round(gpt_seconds, 1), "lstm": round(lstm_seconds, 1)},
    "untrained_gpt_val_loss": round(untrained_loss, 4),
    "results": {
        name: {split: {key: round(v, 4) if isinstance(v, float) else v for key, v in r.items()}
               for split, r in rows_by_split.items()}
        for name, rows_by_split in [("char_trigram", ngram[3]), ("char_5gram", ngram[5]),
                                    ("lstm", results["lstm"]), ("gpt", results["gpt"])]
    },
    "score_std": {str(d_k): [round(a, 3), round(b, 3)] for d_k, (a, b) in score_std.items()},
    "mean_attention_distance": [[round(v, 3) for v in row] for row in distance.tolist()],
    "samples_tau_1": {name: samples[name, 1.0] for name in ("mini-GPT", "LSTM")},
}
with open("lab05_results.json", "w", encoding="utf-8") as f:
    json.dump(card, f, indent=2)

with torch.no_grad():  # the 6 x 6 maps for the briefing figure (briefing section 3) and the passage above
    _, tobe = gpt(torch.tensor([[char_to_id[ch] for ch in "To be,"]], device=DEVICE), return_attention=True)
heads = {
    "To be,": [[A[0, r].cpu().double().numpy().round(4).tolist() for r in range(CONFIG["n_h"])] for A in tobe],
    PASSAGE: [[A[r].double().numpy().round(4).tolist() for r in range(CONFIG["n_h"])] for A in passage_attention],
}
with open("lab05_attention.json", "w", encoding="utf-8") as f:
    json.dump({"quick": QUICK, "layout": "passage -> layer -> head -> row t -> column i", "heads": heads}, f)

TIMES["core path"] = time.time() - LAB_START
print("Run time by section (seconds):")
for section, seconds in TIMES.items():
    print(f"  {section:<26} {seconds:>6.0f}")
print("Wrote lab05_results.json and lab05_attention.json. This is the end of the core path.")

## Stretch (optional) · Write the Multi-Head Layer and the Block Yourself

For those who finish early. Nothing later in the workshop depends on this section; Module 6 loads pretrained transformers through Hugging Face.

**Task.** Without looking back at the provided classes, write two `forward` methods.

- `MyMultiHeadAttention.forward(H, mask)`: project `H` of shape `(B, T, d)` with `self.W_Q`, `self.W_K`, `self.W_V`; split each result into `self.n_h` heads of width `self.d_k` with `view` and `transpose`, giving `(B, n_h, T, d_k)`; call your `scaled_dot_product_attention`; concatenate the heads back to `(B, T, d)`; apply `self.proj` and `self.dropout` (briefing eq. `mha`). Return the output and `A`.
- `MyBlock.forward(H, mask)`: the pre-norm block of briefing eq. `block`, using `self.ln1`, `self.attn`, `self.ln2`, `self.ffn`. Return the new `H` and the attention weights.

The classes inherit their parameters from the provided ones, so the provided model's weights load into yours and the outputs can be compared exactly.

**Predict.** How many parameters does one block have with $n_h = 1$? With $n_h = 8$?

In [ ]:
# TODO 6 (stretch): write the two forward methods.
class MyMultiHeadAttention(MultiHeadAttention):
    def forward(self, H, mask):
        """H: (B, T, d). Returns (output (B, T, d), A (B, n_h, T, T))."""
        raise NotImplementedError("TODO 6")


class MyBlock(Block):
    def __init__(self, d, n_h, d_ff, dropout=0.0):
        super().__init__(d, n_h, d_ff, dropout)
        self.attn = MyMultiHeadAttention(d, n_h, dropout)  # your attention inside your block

    def forward(self, H, mask):
        """Pre-norm block. H: (B, T, d). Returns (H_out (B, T, d), A (B, n_h, T, T))."""
        raise NotImplementedError("TODO 6")

In [ ]:
#@title Solution 6 — try it yourself first { display-mode: "form" }
@workshop.solution(6)
class MyMultiHeadAttention(MultiHeadAttention):
    def forward(self, H, mask):
        """H: (B, T, d). Returns (output (B, T, d), A (B, n_h, T, T))."""
        B, T, d = H.shape

        def split(X):  # (B, T, d) -> (B, n_h, T, d_k): columns r*d_k .. (r+1)*d_k - 1 are head r
            return X.view(B, T, self.n_h, self.d_k).transpose(1, 2)

        Q, K, V = split(self.W_Q(H)), split(self.W_K(H)), split(self.W_V(H))
        V_bar, A = scaled_dot_product_attention(Q, K, V, mask)  # the head is a batch dimension
        V_bar = V_bar.transpose(1, 2).reshape(B, T, d)  # [V_bar^(1); ...; V_bar^(n_h)]
        return self.dropout(self.proj(V_bar)), A


@workshop.solution(6)
class MyBlock(Block):
    def __init__(self, d, n_h, d_ff, dropout=0.0):
        super().__init__(d, n_h, d_ff, dropout)
        self.attn = MyMultiHeadAttention(d, n_h, dropout)  # your attention inside your block

    def forward(self, H, mask):
        """Pre-norm block. H: (B, T, d). Returns (H_out (B, T, d), A (B, n_h, T, T))."""
        attn_out, A = self.attn(self.ln1(H), mask)
        H = H + attn_out  # H' = H + MHA(LN(H)): the residual stream is never normalized
        return H + self.ffn(self.ln2(H)), A  # H_out = H' + FFN(LN(H'))

<details>
<summary>Why this solution works</summary>

`view(B, T, n_h, d_k)` cuts the last dimension of a `(B, T, d)` tensor into $n_h$ consecutive slices of width $d_k$, so columns $r d_k, \dots, (r + 1) d_k - 1$ of the big projection are head $r$'s $W_Q^{(r)}$. `transpose(1, 2)` moves the head next to the batch, and your attention function treats both as batch dimensions. Transposing back and `reshape(B, T, d)` lays the head outputs side by side: the concatenation of eq. `mha`. (`reshape` rather than `view`, because the transpose made the tensor non-contiguous.)

The block is eq. `block` read line by line. Each sublayer reads a normalized copy of the stream and adds its output to the unnormalized stream, so there is an identity path from the block's input to its output.

The parameter count is $3d^2$ for the projections plus $d^2 + d$ for `proj`, whatever $n_h$ is: more heads are more, narrower lookups, not more weights.

</details>

In [ ]:
# Checkpoint 6 (stretch)
workshop.checkpoint(6)
torch.manual_seed(SEED)
_d, _n_h, _T = CONFIG["d"], CONFIG["n_h"], 16
_reference = Block(_d, _n_h, CONFIG["d_ff"]).eval()
_mine = MyBlock(_d, _n_h, CONFIG["d_ff"]).eval()
_mine.load_state_dict(_reference.state_dict())  # the same weights in both
_H = torch.randn(2, _T, _d)
_mask = causal_mask(_T)
with torch.no_grad():
    _ref_out, _ref_A = _reference(_H, _mask)
    _my_out, _my_A = _mine(_H, _mask)
assert _my_out.shape == (2, _T, _d) and _my_A.shape == (2, _n_h, _T, _T)
assert torch.allclose(_my_out, _ref_out, atol=1e-5) and torch.allclose(_my_A, _ref_A, atol=1e-6), \
    "your block differs from the reference with the same weights"

# n_h = 1 is Exercise 1 followed by the projection.
_one = MyMultiHeadAttention(_d, n_h=1).eval()
with torch.no_grad():
    _expected = _one.proj(scaled_dot_product_attention(_one.W_Q(_H), _one.W_K(_H), _one.W_V(_H), _mask)[0])
    assert torch.allclose(_one(_H, _mask)[0], _expected, atol=1e-5), "n_h = 1 must reproduce Exercise 1"

# The leak test of Exercise 2 on your block.
_H_changed = _H.clone()
_H_changed[:, 9] += 1.0
with torch.no_grad():
    _changed_out, _ = _mine(_H_changed, _mask)
assert (_changed_out[:, :9] - _my_out[:, :9]).abs().max() <= 1e-6, "your block reads ahead"
assert (_changed_out[:, 9:] - _my_out[:, 9:]).abs().max() > 1e-3

# Parameter count: the reference's, for every number of heads.
_counts = {n_h: count_parameters(MyBlock(_d, n_h, CONFIG["d_ff"])) for n_h in (1, 2, 4, 8)}
assert set(_counts.values()) == {count_parameters(_reference)}, _counts
print(f"Parameters per block for n_h = 1, 2, 4, 8: {sorted(set(_counts.values()))[0]:,} each")
print("Checkpoint 6 passed: your block matches the reference, does not read ahead, and its size "
      "does not depend on n_h")

In [ ]:
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Which checkpoints passed, and on whose code; then a run record for the instructor.
workshop.summary()
workshop.run_record()

<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
---
**Next:** [6 · Pretraining and the Hugging Face Stack](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/06-pretraining-huggingface.ipynb)

[Workshop site](https://project-delphi.github.io/nlp-llms) · [All notebooks](https://project-delphi.github.io/nlp-llms/notebooks.html) · [Source](https://github.com/project-delphi/nlp-llms)

Genial Labs · text CC BY 4.0, code MIT